# G1-D 双臂绘制山水画

左臂画太阳，右臂画简化山体与飞鸟。执行采用人工确认正确的双臂示教，Astra 持续根据头部右目、左右腕和第三视角驱动任务执行并检查结果。

执行前清空双臂工作区，确认笔已固定、急停可触达、第三视角服务在 5560 端口运行；不要另开 teleop 控制进程。

In [ ]:
from pathlib import Path
import os, subprocess, sys, time

ROOT = Path('/home/karthus_chen/ycb_ws/piper-astra-jev')
TELEOP = Path('/home/karthus_chen/ycb_ws/g1d_teleop')
DEMO = TELEOP / 'data/demo/draw_picture/episode_0001.mcap'
RUN_DIR = ROOT / 'data' / f'g1d_draw_landscape_{time.time_ns()}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
assert DEMO.is_file(), DEMO
assert (TELEOP / 'robot_teleop/config/g1_d.yaml').is_file()
assert os.environ.get('OPENAI_API_KEY') or os.environ.get('UNITREE_OPENAI_API_KEY') or (ROOT / '.env').is_file()
print('示教:', DEMO, '| 运行记录:', RUN_DIR)

## 启动闭环

确认机器人已处于允许自动运动的状态后运行下一格。UI 的 Q/Esc 可请求停止；急停优先级最高。

In [ ]:
env = os.environ.copy()
env['G1D_TELEOP_ROOT'] = str(TELEOP)
ui = subprocess.Popen([sys.executable, str(ROOT/'scripts/ui.py'), str(RUN_DIR)], cwd=ROOT, env=env)
time.sleep(2)
agent = subprocess.Popen([sys.executable, str(ROOT/'scripts/g1d_reference_agent.py'),
    '--run-dir', str(RUN_DIR), '--demo', str(DEMO), '--net', 'enp4s0',
    '--task-text', '左臂画太阳，右臂画简化山体与飞鸟',
    '--agent-instructions', '左臂画太阳，右臂画简化山体轮廓和线条飞鸟；根据四路现场画面检查安全和最终画面。',
    '--phase-label', '双臂山水画', '--segment-frames', '25', '--slowdown', '2.0'],
    cwd=ROOT, env=env)
try:
    result = agent.wait()
finally:
    if agent.poll() is None: agent.terminate()
    if ui.poll() is None: ui.terminate()
print('agent exit:', result, '| status:', (RUN_DIR/'status.json').read_text() if (RUN_DIR/'status.json').exists() else 'missing')
print('录像:', RUN_DIR/'operator_view.mp4')